In [ ]:
# ── Setup — run this cell first ──────────────────────────────────────────────
import os
import subprocess

FOLDER_ID = "1rSePBwwgEaTwP2JmWi4SxMeMcYYjY8M6"
BASE_DIR = "/content/strata_forced_data/"
os.makedirs(BASE_DIR, exist_ok=True)

# Try downloading with gdown, with retries
!pip install -q gdown

import time
max_retries = 3
for attempt in range(max_retries):
    try:
        print(f"Attempting download (attempt {attempt + 1}/{max_retries})...")
        import gdown
        gdown.download_folder(id=FOLDER_ID, output=BASE_DIR, quiet=False)
        print("✓ Download succeeded")
        break
    except Exception as e:
        print(f"✗ Attempt {attempt + 1} failed: {e}")
        if attempt < max_retries - 1:
            print(f"  Waiting 10 seconds before retry...")
            time.sleep(10)
        else:
            print("\n⚠ Download failed after all retries.")
            print("Trying alternative method with curl...")
            # Fallback: use curl to download the folder as a zip
            try:
                url = f"https://drive.google.com/uc?id={FOLDER_ID}&export=download"
                result = subprocess.run(
                    ["curl", "-L", "-o", f"{BASE_DIR}/data.zip", url],
                    capture_output=True, timeout=300
                )
                if result.returncode == 0:
                    print("Downloaded as zip, extracting...")
                    subprocess.run(["unzip", "-q", f"{BASE_DIR}/data.zip", "-d", BASE_DIR])
                    print("✓ Extraction succeeded")
            except Exception as curl_error:
                print(f"Curl also failed: {curl_error}")

# Find data directory
import glob
label_files = glob.glob(BASE_DIR + "/**/labels_*.npy", recursive=True)
print(f"\nFound {len(label_files)} label files")
if label_files:
    DATA_DIR = os.path.dirname(label_files[0])
    print(f"DATA_DIR = {DATA_DIR}")
else:
    print("ERROR: No label files found!")
    print("Checking what's in BASE_DIR:")
    for root, dirs, files in os.walk(BASE_DIR):
        level = root.replace(BASE_DIR, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        if level < 2:
            for f in files[:5]:
                print(f'{indent}  {f}')

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from matplotlib.colors import ListedColormap
import glob
import re
import math
import os

# ── Validate DATA_DIR ────────────────────────────────────────────────────────
print("=" * 60)
print("VALIDATION CHECK")
print("=" * 60)

if 'DATA_DIR' not in globals():
    raise RuntimeError(
        "\n❌ ERROR: DATA_DIR is not defined!\n\n"
        "You must run the SETUP cell first to:\n"
        "  1. Download data from Google Drive\n"
        "  2. Define the DATA_DIR path\n\n"
        "Please scroll up and run the 'Setup' cell, then re-run this cell."
    )

print(f"✓ DATA_DIR defined: {DATA_DIR}")

if not os.path.exists(DATA_DIR):
    raise RuntimeError(
        f"\n❌ ERROR: DATA_DIR does not exist!\n\n"
        f"Path: {DATA_DIR}\n\n"
        f"The setup cell may have failed to download data.\n"
        f"Check your Google Drive access and re-run the setup cell."
    )

print(f"✓ DATA_DIR exists\n")

# ── Config ────────────────────────────────────────────────────────────────────
DS       = 8
TSTAMP   = "0.472384"
DGRAD    = -35.9647
tag      = f"ds{DS}"
_NxD, _NyD, _NzD = 4096 // DS, 2048 // DS, 4096 // DS

# ── Scan available label files ───────────────────────────────────────────────
print("Scanning for label files...")
_label_files = sorted(glob.glob(os.path.join(DATA_DIR, f"labels_{tag}_sigma*_k*.npy")))

if not _label_files:
    print(f"\n❌ ERROR: No label files found!\n")
    print(f"Looking for: labels_{tag}_sigma<N>_k<M>.npy")
    print(f"In directory: {DATA_DIR}\n")
    print("Files in DATA_DIR:")
    try:
        all_files = sorted(os.listdir(DATA_DIR))
        if all_files:
            for f in all_files[:10]:
                size = os.path.getsize(os.path.join(DATA_DIR, f))
                print(f"  {f} ({size/1e6:.1f} MB)")
            if len(all_files) > 10:
                print(f"  ... and {len(all_files) - 10} more files")
        else:
            print("  (directory is empty)")
    except Exception as e:
        print(f"  Could not list directory: {e}")
    raise RuntimeError(
        f"\nThe setup cell did not download data successfully.\n"
        f"Make sure you have access to Google Drive folder ID:\n"
        f"  1rSePBwwgEaTwP2JmWi4SxMeMcYYjY8M6\n\n"
        f"Re-run the setup cell and check for errors."
    )

_combos = []
for _lf in _label_files:
    _m = re.search(r'sigma(\d+)_k(\d+)\.npy', _lf)
    if _m:
        _combos.append((int(_m.group(1)), int(_m.group(2))))

if not _combos:
    raise RuntimeError(f"ERROR: Could not parse sigma/k values from label files")

print(f"✓ Found {len(_combos)} clustering combinations")
print(f"  Examples: {_combos[:3]}\n")

SIGMA, K = _combos[0]

# ── Check for required data files ────────────────────────────────────────────
print("Checking for required data files...")
required = [
    f"drdy_{tag}_norm.npy",
    f"log_eps_{tag}.npy",
    f"log_chi_{tag}.npy",
    f"r_{tag}.npy",
    f"binary_filtered_{tag}_sigma{SIGMA}.npy",
]

missing = [f for f in required if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    print(f"\n⚠️  WARNING: {len(missing)} expected files not found:\n")
    for f in missing:
        print(f"  - {f}")
    raise RuntimeError(
        f"\nRequired data files are missing.\n"
        f"The setup cell may not have downloaded the complete dataset.\n"
        f"Check the setup cell output for errors."
    )

print(f"✓ All {len(required)} required data files present\n")
print("=" * 60)
print("LOADING DATA")
print("=" * 60 + "\n")

# ── Load data ────────────────────────────────────────────────────────────────
_drdy_arr = np.load(os.path.join(DATA_DIR, f"drdy_{tag}_norm.npy"))
print(f"✓ drdy: shape {_drdy_arr.shape}")

_log_eps = np.load(os.path.join(DATA_DIR, f"log_eps_{tag}.npy"))
_log_chi = np.load(os.path.join(DATA_DIR, f"log_chi_{tag}.npy"))
print(f"✓ log_eps: shape {_log_eps.shape}")
print(f"✓ log_chi: shape {_log_chi.shape}")

_leps_vmin = float(np.nanpercentile(_log_eps[:, :, _NzD // 2], 2))
_leps_vmax = float(np.nanpercentile(_log_eps[:, :, _NzD // 2], 98))
_lchi_vmin = float(np.nanpercentile(_log_chi[:, :, _NzD // 2], 2))
_lchi_vmax = float(np.nanpercentile(_log_chi[:, :, _NzD // 2], 98))

def _drdy_slicer(axis, idx):
    if axis == "z":
        sl = _drdy_arr[:, :, idx]
    elif axis == "y":
        sl = _drdy_arr[:, idx, :]
    else:
        sl = _drdy_arr[idx, :, :]
    return (sl + DGRAD) / np.abs(DGRAD)

# ── Cache for loaded arrays ───────────────────────────────────────────────────
_bf_cache  = {}
_lbl_cache = {}

def _load_clustering(sigma, k):
    if sigma not in _bf_cache:
        print(f"  Loading binary_filtered sigma={sigma}...")
        _bf_cache[sigma] = np.load(os.path.join(DATA_DIR, f"binary_filtered_{tag}_sigma{sigma}.npy"))
    if (sigma, k) not in _lbl_cache:
        print(f"  Loading labels sigma={sigma} k={k}...")
        _lbl_cache[(sigma, k)] = np.load(os.path.join(DATA_DIR, f"labels_{tag}_sigma{sigma}_k{k}.npy"))
    return _bf_cache[sigma], _lbl_cache[(sigma, k)]

print(f"\nLoading initial clustering (σ={SIGMA}, k={K})...")
_bf0, _lbl0 = _load_clustering(SIGMA, K)

# ── Load the density field once ──────────────────────────────────────────────
_r_arr = np.load(os.path.join(DATA_DIR, f"r_{tag}.npy"))
print(f"✓ r: shape {_r_arr.shape}")
print(f"✓ Data loaded\n")

# ── Fields ───────────────────────────────────────────────────────────────────
FIELDS = [
    dict(data=_r_arr,   title=r"Turbulent Density $\rho$", cmap="RdBu_r", vmin=-10, vmax=10),
    dict(slicer=_drdy_slicer, shape=(_NxD,_NyD,_NzD), title=r"$\partial \rho/\partial y + \alpha$", cmap="RdBu_r", vmin=-10, vmax=10),
    dict(data=_bf0,     title=f"Filtered overturning fraction (σ={SIGMA/DS:.4g})", cmap="RdBu_r", vmin=0, vmax=0.5),
    dict(data=_lbl0,    title=f"K-means clustering (k={K})"),
    dict(data=_log_eps, title=r"$\log_{10}(\varepsilon)$", cmap="inferno", vmin=_leps_vmin, vmax=_leps_vmax),
    dict(data=_log_chi, title=r"$\log_{10}(\chi)$",        cmap="inferno", vmin=_lchi_vmin, vmax=_lchi_vmax),
]

_OVERLAY_SRC    = 3        # clustering field provides the overlay contours
_OVERLAY_TGTS   = list(range(len(FIELDS)))
_OVERLAY_COLORS = {0: "black", 1: "black"}
_SIGMA_PANEL    = 2        # panel that shows the sigma circle

_AXIS_CFG = {
    "z": {"dim": 2, "xl": "$x$", "yl": "$y$", "T": True},
    "y": {"dim": 1, "xl": "$x$", "yl": "$z$", "T": True},
    "x": {"dim": 0, "xl": "$z$", "yl": "$y$", "T": False},
}

# shared mutable state, driven by the widgets in the next cell
_state = {"axis": "z", "idx": 0, "overlay": False, "sigma": SIGMA, "k": K}

def _get_slice(f, axis, idx):
    if "slicer" in f:
        return f["slicer"](axis, idx)
    arr = f["data"]
    if axis == "x":   return arr[idx]
    elif axis == "y": return arr[:, idx]
    else:             return arr[:, :, idx]

def _field_shape(f):
    return f["shape"] if "slicer" in f else f["data"].shape

def _discrete_cmap(labels):
    """Return (cmap, vmin, vmax, ticks) for an integer label array."""
    n = int(labels.max()) + 1
    cmap = ListedColormap(plt.get_cmap("tab10").colors[:n])
    return cmap, -0.5, n - 0.5, list(range(n))

# ── Layout constants ──────────────────────────────────────────────────────────
_PANEL_W  = 4.5
_N_COLS   = 3
_N_FIELDS = len(FIELDS)
_N_ROWS   = math.ceil(_N_FIELDS / _N_COLS)

def _make_figure(axis, idx):
    """Build the full 6-panel figure from scratch for the given axis/index.

    Rebuilding every time is the most robust approach on Colab's inline
    backend (no canvas/draw_idle tricks that silently fail).
    """
    cfg  = _AXIS_CFG[axis]
    do_T = cfg["T"]

    # figure size from the first field's slice aspect ratio
    _sl0     = _get_slice(FIELDS[0], axis, idx)
    _h0, _w0 = _sl0.shape
    panel_h  = _PANEL_W * (_w0 / _h0) if do_T else _PANEL_W * (_h0 / _w0)

    fig, axs2d = plt.subplots(
        _N_ROWS, _N_COLS,
        figsize=(_PANEL_W * _N_COLS, panel_h * _N_ROWS + 0.9),
        squeeze=False,
    )
    plt.subplots_adjust(left=0.05, right=0.97, bottom=0.08, top=0.90,
                        wspace=0.25, hspace=0.15)

    axs = [axs2d[r][c] for r in range(_N_ROWS) for c in range(_N_COLS)]
    for ax in axs[_N_FIELDS:]:
        ax.set_visible(False)
    axs = axs[:_N_FIELDS]

    last_ext = None
    for i, (ax, f) in enumerate(zip(axs, FIELDS)):
        sl   = _get_slice(f, axis, idx)
        h, w = sl.shape
        if do_T:
            data = np.ma.masked_invalid(sl.T)
            ext  = [-0.5, h - 0.5, -0.5, w - 0.5]
        else:
            data = np.ma.masked_invalid(sl)
            ext  = [-0.5, w - 0.5, -0.5, h - 0.5]
        last_ext = ext

        # the clustering panel uses a discrete colormap sized to k
        if i == _OVERLAY_SRC:
            cmap, vmin, vmax, ticks = _discrete_cmap(f["data"])
        else:
            cmap  = f.get("cmap", "viridis")
            vmin  = f.get("vmin")
            vmax  = f.get("vmax")
            ticks = f.get("ticks")

        kw = dict(origin="lower", aspect="equal", cmap=cmap,
                  interpolation="nearest", extent=ext)
        if vmin is not None: kw["vmin"] = vmin
        if vmax is not None: kw["vmax"] = vmax

        im = ax.imshow(data, **kw)
        plt.colorbar(im, ax=ax, shrink=0.8, ticks=ticks)
        ax.set_title(f.get("title", ""), fontsize=9)
        ax.set_xlabel(cfg["xl"])
        ax.set_ylabel(cfg["yl"])

    # ── sigma circle on the filtering panel ───────────────────────────────────
    sd  = _state["sigma"] / DS
    pad = sd * 0.4
    axs[_SIGMA_PANEL].add_patch(
        plt.Circle((sd + pad, sd + pad), sd, fill=False,
                   edgecolor='green', linewidth=3, zorder=5)
    )
    axs[_SIGMA_PANEL].text(sd + pad, sd + pad, r'$\sigma$', color='green',
                           ha='center', va='center', fontsize=7, zorder=6)

    # ── cluster-border overlay ────────────────────────────────────────────────
    if _state["overlay"]:
        lbl_sl = _get_slice(FIELDS[_OVERLAY_SRC], axis, idx)
        kk     = int(FIELDS[_OVERLAY_SRC]["data"].max()) + 1
        levels = np.arange(0.5, kk)
        ld     = lbl_sl.T if do_T else lbl_sl
        for ti in _OVERLAY_TGTS:
            axs[ti].contour(ld, levels=levels,
                            colors=_OVERLAY_COLORS.get(ti, "white"),
                            linewidths=0.6, alpha=0.7, extent=last_ext)

    fig.suptitle(
        f"{axis}-slice {idx}  "
        f"(σ={_state['sigma']}, k={_state['k']}, every {DS} grid pts)",
        fontsize=11,
    )
    return fig

print("✓ Visualization function ready!\n")
print("Run the next cell to display the interactive viewer.")

In [ ]:
# ── Interactive controls (ipywidgets + inline figure rebuild) ─────────────────
# The figure is rebuilt from scratch into an Output widget on every change.
# This is the robust Colab pattern (no %matplotlib widget / ipympl needed).

axis_dropdown = widgets.Dropdown(
    options=['z', 'y', 'x'], value='z',
    description='Slice axis:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='250px'),
)

slider = widgets.IntSlider(
    value=0, min=0, max=_field_shape(FIELDS[0])[2] - 1, step=1,
    description='Slice index:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px'),
)

overlay_toggle = widgets.Checkbox(
    value=False, description='Show cluster borders',
    style={'description_width': 'initial'},
)

combo_dd = widgets.Dropdown(
    options=[(f"σ={s}, k={k}", (s, k)) for s, k in _combos],
    value=_combos[0],
    description='Clustering:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='250px'),
)

out = widgets.Output()

# guard to suppress the slider's callback while we adjust it programmatically
_suspend = {"v": False}

def _refresh():
    with out:
        out.clear_output(wait=True)
        fig = _make_figure(_state["axis"], slider.value)
        plt.show()
        plt.close(fig)

def _on_axis_change(change):
    if change["name"] != "value":
        return
    axis = change["new"]
    _state["axis"] = axis
    n = _field_shape(FIELDS[0])[_AXIS_CFG[axis]["dim"]]
    _suspend["v"] = True
    slider.max = n - 1
    if slider.value > n - 1:
        slider.value = n - 1
    _suspend["v"] = False
    _refresh()

def _on_slider_change(change):
    if change["name"] != "value" or _suspend["v"]:
        return
    _state["idx"] = int(change["new"])
    _refresh()

def _on_overlay_toggle(change):
    if change["name"] != "value":
        return
    _state["overlay"] = change["new"]
    _refresh()

def _on_combo_change(change):
    if change["name"] != "value":
        return
    sigma, k = change["new"]
    bf, lbl = _load_clustering(sigma, k)
    FIELDS[2]["data"]  = bf
    FIELDS[2]["title"] = f"Filtered overturning fraction (σ={sigma/DS:.4g})"
    FIELDS[3]["data"]  = lbl
    FIELDS[3]["title"] = f"K-means clustering (k={k})"
    _state["sigma"] = sigma
    _state["k"]     = k
    _refresh()

axis_dropdown.observe(_on_axis_change, names='value')
slider.observe(_on_slider_change, names='value')
overlay_toggle.observe(_on_overlay_toggle, names='value')
combo_dd.observe(_on_combo_change, names='value')

controls = widgets.VBox([combo_dd, axis_dropdown, slider, overlay_toggle])
display(controls, out)

# draw the initial frame
_refresh()